# Generic Utils
> This module handles all data/model/trainer loading logic based on the pipeline config. It provides a clean interface for the main training script, abstracting away the details of how different components are instantiated.

In [ ]:
#| default_exp utils.__init__

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fastcore import *
from fastcore.utils import *

In [ ]:
#| export
from omegaconf import OmegaConf, DictConfig
import hydra
import torch

In [ ]:
#| export
def init_data(cfg: DictConfig):
    """Instantiates the correct datamodule based on the pipeline config."""
    print(f"Initializing Datamodule: {cfg.pipeline.datamodule._target_}")
    return hydra.utils.instantiate(cfg.pipeline.datamodule)


In [ ]:
#| export
def init_model(cfg: DictConfig):
    """
    Instantiates the model(s).
    Returns a single model for Stage 1, or a dictionary of 3 models for Stage 2.
    """
    stage = cfg.pipeline.stage
    model_cfg = cfg.pipeline.model

    if stage == 1:
        # Stage 1: Simply instantiate and return the single VQ-VAE model
        return hydra.utils.instantiate(model_cfg)

    elif stage == 2:
        # Stage 2: Instantiate all three models independently
        models = {
            "power_net": hydra.utils.instantiate(model_cfg.power_net),
            "jepa": hydra.utils.instantiate(model_cfg.jepa)
        }
        
        # Instantiate VQ-VAE, then immediately load its pretrained weights and freeze it
        vqvae = hydra.utils.instantiate(model_cfg.vqvae)
        
        if hasattr(model_cfg.vqvae, "checkpoint_path") and model_cfg.vqvae.checkpoint_path:
            print(f"Loading pretrained VQ-VAE weights from {model_cfg.vqvae.checkpoint_path}")
            checkpoint = torch.load(model_cfg.vqvae.checkpoint_path, map_location="cpu")
            vqvae.load_state_dict(checkpoint["model_state_dict"]) 
            
        # Freeze VQ-VAE because it's only used for getting latent codebooks
        vqvae.eval()
        for param in vqvae.parameters():
            param.requires_grad = False
            
        models["vqvae"] = vqvae
        return models
    
    else:
        raise ValueError(f"Unknown pipeline stage: {stage}")



In [ ]:
#| export
def init_trainer(cfg: DictConfig, data_module, models, device):
    """Instantiates the trainer and injects the loaded models and data."""
    # We pass models and datamodule directly into the instantiation call 
    # so the trainer receives its dependencies right away.
    return hydra.utils.instantiate(
        cfg.pipeline.trainer, 
        data_module=data_module,
        model=models, 
        device=device
    )


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()